# 📝 데이터베이스 기초 과제 LV3 정답 — 여행지 추천 RAG (강사용)

단계별 **모범답안 + 해설**입니다. 학생이 스스로 푼 뒤 비교하도록 안내하세요.

- 경로는 정답 노트북 기준이라 `.env` 는 `../../day17_데이터베이스_SQL/.env` 를 함께 읽습니다.
- 데이터는 교안과 **일부러 다릅니다** — 교안_03 은 연구비 FAQ, 이 과제는 여행지 100곳입니다. 배운 절차를 새 데이터에 옮기는 것이 이 과제의 목적입니다.
- 유사도 기대값은 같은 임베딩 모델로 **로컬에서 계산해 얼린 실측값**입니다.
- 문턱값 **0.35** 은 실측 분포에서 골랐습니다 — 통과해야 할 질문과 0.16, 걸러야 할 질문과 0.20 만큼 떨어져 있습니다.
- 검색 대상 텍스트는 **"이름 + 공백 + 설명"** 입니다. 적재할 때와 다르게 만들면 **에러 없이 순위만 틀어집니다** — 1단계 자가채점이 실제 검색으로 이것을 되짚습니다.
- 문제 2 의 자가채점은 생성된 **문장을 채점하지 않습니다** — 출처 표기·거절 여부처럼 프로그램이 지켜야 할 계약만 봅니다.

---

## 준비 — 실행만 하세요

교안_03 에서 쓴 것과 **같은 준비 셀**입니다. 연결과 임베딩 모델을 준비합니다.

In [ ]:
# [제공 코드] — Supabase 연결 준비 (내용은 이해하지 않아도 됩니다 — 실행만 하세요)
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

ROOT = Path(".") if Path("data").is_dir() else Path("..")
load_dotenv(ROOT / ".env")

project_url = os.getenv("SUPABASE_URL")
anon_key = os.getenv("SUPABASE_ANON_KEY")
if not project_url or not anon_key:
    raise RuntimeError(
        "Supabase 연결 정보를 찾지 못했습니다 — SUPABASE_URL / SUPABASE_ANON_KEY 가 비어 있습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) Supabase 대시보드 -> Project Settings -> API 에서\n"
        "     Project URL 과 anon public 키를 복사해 .env 에 붙여넣으세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

supabase = create_client(project_url, anon_key)


def to_df(response):
    """supabase 응답의 .data(딕셔너리 목록)를 pandas DataFrame 으로 바꿉니다."""
    return pd.DataFrame(response.data)


print("Supabase 연결 준비 완료 —", project_url)

In [ ]:
# [제공 코드] — 임베딩 모델 준비 (지난 단원에서 쓴 그 모델입니다 — 실행만 하세요)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("jhgan/ko-sroberta-multitask")


def embed(text):
    """질문 한 문장을 768개 숫자(파이썬 리스트)로 바꿉니다."""
    return model.encode([text], normalize_embeddings=True)[0].tolist()


print("임베딩 차원:", model.get_embedding_dimension())

---

# 문제 1. 여행지를 담고, 뜻으로 찾기

**배경**: 여행지를 고를 때 사람은 이름으로 찾지 않습니다. "바다를 보며 쉴 수 있는 곳" 처럼 **하고 싶은 것**으로 찾습니다. 소개글과 뜻이 가까운 곳을 찾아 주려면 임베딩이 필요하고, "부산에서" 같은 조건까지 함께 걸려면 그 벡터가 **표 안에** 있어야 합니다.

## 먼저 — 표와 검색 함수를 만듭니다

교안_03 에서 본 대로, 표·인덱스·함수를 만드는 일(**DDL**)은 supabase-py 로 보낼 수 없습니다. **Supabase 대시보드 → SQL Editor** 를 열고 아래를 **통째로 붙여넣어 Run** 하세요(마지막 줄 `$$;` 까지). 같은 내용이 **`data/setup_travel.sql`** 에도 있으니 그 파일을 열어 복사해도 됩니다.

```sql
-- day17 과제 LV3 준비 SQL — Supabase 대시보드 -> SQL Editor 에 통째로 붙여넣고 Run 하세요.
-- 언제 몇 번을 다시 실행해도 안전합니다(표를 지우고 다시 만듭니다).
-- 데이터: 여행지 100곳은 수업용 가상 데이터입니다(15일차 실습자료에서 가져왔습니다).

-- 1) 확장 켜기 — vector 타입과 거리 연산자(<=>)가 생깁니다.
CREATE EXTENSION IF NOT EXISTS vector;

-- 2) 실습 표 정리 (자식 -> 부모 순서)
DROP TABLE IF EXISTS travel_docs CASCADE;
DROP TABLE IF EXISTS travel_region CASCADE;

-- 3) 지역과 권역 표 (JOIN 대상) — 지역 15종이 어느 행정 권역에 속하는지
CREATE TABLE travel_region (
    region varchar(10) PRIMARY KEY,
    area   varchar(10) NOT NULL
);

INSERT INTO travel_region (region, area) VALUES
    ('강원', '관동'),
    ('경기', '수도권'),
    ('경남', '영남'),
    ('경북', '영남'),
    ('광주', '호남'),
    ('대구', '영남'),
    ('대전', '충청'),
    ('부산', '영남'),
    ('서울', '수도권'),
    ('울산', '영남'),
    ('인천', '수도권'),
    ('전남', '호남'),
    ('전북', '호남'),
    ('제주', '제주'),
    ('충남', '충청');

-- 4) 여행지 본문 + 임베딩 표 (768차원 = jhgan/ko-sroberta-multitask)
--    CSV 의 type 열은 이 표에서 spot_type 이라는 이름으로 들어갑니다.
CREATE TABLE travel_docs (
    spot_id      varchar(10) PRIMARY KEY,
    name         varchar(40) NOT NULL,
    region       varchar(10) NOT NULL REFERENCES travel_region(region),
    spot_type    varchar(10) NOT NULL,
    entrance_fee int NOT NULL,
    description  text NOT NULL,
    embedding    vector(768)
);

-- 5) 벡터 인덱스 — 코사인 거리(<=>)용 HNSW
CREATE INDEX travel_docs_embedding_idx
    ON travel_docs USING hnsw (embedding vector_cosine_ops);

-- 6) 의미 검색 함수 — 노트북에서 supabase.rpc("match_spot", {...}) 로 부릅니다.
--    filter_region 과 filter_type 은 NULL 이면 그 조건을 걸지 않습니다.
--    min_similarity 는 그 값보다 가까운 것만 남깁니다(기본 0 이면 전부 통과).
--    travel_region 을 JOIN 해서 권역(area)까지 함께 돌려줍니다.
CREATE OR REPLACE FUNCTION match_spot (
    query_embedding vector(768),
    match_count     int   DEFAULT 3,
    filter_region   text  DEFAULT NULL,
    filter_type     text  DEFAULT NULL,
    min_similarity  float DEFAULT 0
)
RETURNS TABLE (
    spot_id      varchar(10),
    name         varchar(40),
    region       varchar(10),
    area         varchar(10),
    spot_type    varchar(10),
    entrance_fee int,
    description  text,
    similarity   float
)
LANGUAGE sql STABLE
AS $$
    SELECT d.spot_id,
           d.name,
           d.region,
           g.area,
           d.spot_type,
           d.entrance_fee,
           d.description,
           1 - (d.embedding <=> query_embedding) AS similarity
    FROM travel_docs d
    JOIN travel_region g ON g.region = d.region
    WHERE (filter_region IS NULL OR d.region = filter_region)
      AND (filter_type IS NULL OR d.spot_type = filter_type)
      AND 1 - (d.embedding <=> query_embedding) >= min_similarity
    ORDER BY d.embedding <=> query_embedding
    LIMIT match_count;
$$;
```

**이 SQL 이 만드는 것**

| | 무엇 | 왜 |
|---|---|---|
| 3 | `travel_region` 표 + 15행 | 지역이 어느 **권역**에 속하는지 (`JOIN` 대상). 이 표는 준비 SQL 이 값까지 채워 줍니다 |
| 4 | `travel_docs` 표 | 소개글 옆에 `embedding vector(768)` 열을 함께 둡니다. `region` 에 **외래키**가 걸려 있어 `travel_region` 에 없는 지역은 들어가지 않습니다 |
| 5 | HNSW 인덱스 | 벡터가 많아져도 전부 훑지 않습니다 |
| 6 | `match_spot(...)` 함수 | 조건으로 좁히고 뜻으로 줄 세우며, `travel_region` 을 **`JOIN`** 해 권역까지 붙여 옵니다 |

**`match_spot` 의 인자 다섯 개** — 문제 1·2 에서 계속 씁니다.

| 인자 | 무엇 | 안 쓰려면 |
|---|---|---|
| `query_embedding` | 질문을 바꾼 벡터(숫자 768개 리스트) | (필수) |
| `match_count` | 최대 몇 곳 | (필수) |
| `filter_region` | 어느 지역 안에서 | `None` |
| `filter_type` | 어느 유형 안에서 | `None` |
| `min_similarity` | 이 값보다 가까운 곳만 | `0.0` |

**돌려주는 것**: 한 곳이 딕셔너리 하나인 리스트입니다. 키는 `spot_id` · `name` · `region` · **`area`** · `spot_type` · `entrance_fee` · `description` · `similarity` 이고, **유사도가 높은 순**으로 옵니다.

> `area`(권역)는 `travel_docs` 에 없는 열입니다 — 함수 안의 `JOIN` 이 `travel_region` 에서 붙여 온 것입니다. 지역 15종이 권역 6종(관동 · 수도권 · 영남 · 제주 · 충청 · 호남)으로 묶입니다.

> CSV 의 `type` 열은 표에서 **`spot_type`** 이라는 이름으로 들어갑니다. 적재할 때 이름을 바꿔 주어야 합니다.

In [ ]:
# [제공 코드] — 위 준비 SQL 을 Run 했는지 확인합니다 (안 돼 있으면 여기서 멈춥니다)
try:
    regions = supabase.table("travel_region").select("*").order("region").execute()
    supabase.table("travel_docs").select("spot_id").limit(1).execute()
except Exception as error:
    raise RuntimeError(
        "travel_region / travel_docs 표를 찾지 못했습니다 - 준비 SQL 을 아직 실행하지 않은\n"
        "  것 같습니다. Supabase 대시보드 -> SQL Editor 를 열고 data/setup_travel.sql 을\n"
        "  통째로 붙여넣어 Run 한 뒤, 이 셀을 다시 실행하세요.\n"
        f"  (원래 에러: {error})") from error

if len(regions.data) != 15:
    raise RuntimeError(
        f"travel_region 이 15행이어야 하는데 {len(regions.data)}행입니다 - "
        "SQL Editor 에서 data/setup_travel.sql 을 처음부터 다시 Run 하세요.")

# 검색 함수까지 만들어졌는지 — 빈 벡터로 한 번 불러 봅니다.
# 표가 아직 비어 있어도 함수가 있으면 빈 목록이 오고, 함수가 없으면 여기서 에러가 납니다.
try:
    supabase.rpc("match_spot", {"query_embedding": [0.0] * 768, "match_count": 1,
                                "filter_region": None, "filter_type": None,
                                "min_similarity": 0.0}).execute()
except Exception as error:
    raise RuntimeError(
        "match_spot 함수를 찾지 못했습니다. 아래 순서로 확인하세요.\n"
        "  1) 방금 Run 했다면 몇 초 기다렸다가 이 셀만 다시 실행하세요 -\n"
        "     새로 만든 함수가 보이기까지 잠깐 걸립니다(SQL Editor 에 성공이라고 떴다면 이 경우).\n"
        "  2) 그래도 같으면 준비 SQL 의 5번(CREATE OR REPLACE FUNCTION match_spot ...)까지\n"
        "     마지막 줄  $$;  을 포함해 끝까지 복사한 뒤 다시 Run 하세요.\n"
        f"  (원래 에러: {error})") from error

print("준비 확인 완료 — travel_region", len(regions.data), "행 · travel_docs 표 · match_spot 함수")
display(to_df(regions))

### 1단계. 여행지를 임베딩해 표에 담기

`data/travel_spots.csv` 을 읽어 **100곳 전부**를 `travel_docs` 에 넣으세요. 소개글을 임베딩한 벡터를 같은 행에 함께 담습니다.

**검색 대상 텍스트는 `이름 + 공백 + 설명` 입니다.** 예: `"해운대 해수욕장 부산을 대표하는 넓은 백사장으로 ..."` — 이름도 함께 넣어야 **이름으로도** 찾힙니다 — "감천문화마을" 이라고만 물었을 때 그곳이 1위로 나오려면 이름이 벡터에 들어 있어야 합니다.

**표에 넣을 열** (CSV 열 이름과 다른 곳이 두 군데입니다)

| 표의 열 | CSV 에서 |
|---|---|
| `spot_id` | `id` |
| `name` · `region` · `description` | 같은 이름 |
| `spot_type` | **`type`** |
| `entrance_fee` | `entrance_fee` — **정수로 바꿔서** 넣습니다 |
| `embedding` | 위에서 만든 벡터(파이썬 리스트 그대로) |

**확인**: `travel_docs` 가 100행이 되고, 지역은 15종, 유형은 5종이 들어 있습니다.

> **적재할 때와 검색할 때의 텍스트가 다르면 에러가 나지 않고 순위만 조용히 틀어집니다.** 자가채점이 실제 검색을 한 번 돌려 이것을 되짚습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 여러 번 실행해도 같은 행이 쌓이지 않게, 넣기 전에 표를 먼저 비운다
- 임베딩은 한 줄씩 부르지 말고 목록을 통째로 한 번에 넘기는 편이 훨씬 빠르다
- 한 번에 백 건을 보내면 요청이 크다. 스무 건 안팎으로 끊어 보낸다
- 조건 없는 삭제는 부를 수 없다. 항상 참인 조건을 하나 준다

세부구현:
1. csv 로 파일을 읽어 딕셔너리 목록으로 만든다
2. 이름과 설명을 이어 붙인 문자열 목록을 만들어 한 번에 임베딩한다
3. 표의 열 이름에 맞춰 딕셔너리 목록을 만든다
   3-1. type 은 spot_type 으로 이름을 바꾼다
   3-2. 입장료는 정수로 바꾼다
   3-3. 벡터는 파이썬 리스트로 바꿔 담는다
4. 표를 비운 뒤 스무 건씩 끊어 넣는다
5. 몇 건이 들어갔는지 세어 출력한다
```

</details>

In [ ]:
# 1) 원본을 읽어 '이름 + 설명' 을 통째로 임베딩합니다. 1분쯤 걸립니다.
import csv

with open(ROOT / "data" / "travel_spots.csv", encoding="utf-8") as f:
    spots = list(csv.DictReader(f))

# 이름을 함께 넣어야 '경복궁' 같은 이름으로도 찾힙니다
texts = [f"{s['name']} {s['description']}" for s in spots]
vectors = model.encode(texts, normalize_embeddings=True)

# 2) 표의 열 이름에 맞춰 담습니다 - type 은 spot_type, 입장료는 정수
records = [
    {"spot_id": s["id"], "name": s["name"], "region": s["region"],
     "spot_type": s["type"], "entrance_fee": int(s["entrance_fee"]),
     "description": s["description"], "embedding": vector.tolist()}
    for s, vector in zip(spots, vectors)
]

# 3) 먼저 비우고 스무 건씩 나눠 보냅니다(한 번에 보내기엔 요청이 큽니다)
# .neq() 로 항상 참인 조건을 줍니다 - 조건 없는 삭제는 부를 수 없습니다
supabase.table("travel_docs").delete().neq("spot_id", "").execute()

for start in range(0, len(records), 20):
    supabase.table("travel_docs").insert(records[start:start + 20]).execute()

loaded = supabase.table("travel_docs").select("spot_id").execute().data
print("적재 완료:", len(loaded), "곳 / 보낸 행:", len(records))

<details><summary>해설</summary>

- `model.encode(목록)` 은 100건을 **한 번에** 처리합니다. 한 줄씩 `embed()` 를 부르면 같은 일을 100번 나눠 하는 셈이라 훨씬 느립니다.
- **벡터는 그냥 파이썬 리스트로 보냅니다** — PostgreSQL 이 `vector(768)` 로 받습니다. `numpy` 배열 그대로는 보낼 수 없어 `.tolist()` 로 바꿉니다.
- 넣기 전에 비우는 이유: 이 셀을 다시 실행해도 같은 행이 두 번 쌓이지 않게 하기 위해서입니다. `spot_id` 가 기본키라 두 번째 실행은 사실 키 충돌로 실패합니다 — 비우고 시작하면 그 일이 없습니다.
- `.delete()` 에 `.neq("spot_id", "")` 를 붙인 것은 **"전부"** 라는 뜻입니다. 조건 없는 `.delete()` 는 부를 수 없기 때문입니다(교안_03 에서는 `.gte()` 를 썼습니다).

</details>

In [ ]:
# [자가채점]
_rows = supabase.table("travel_docs").select(
    "spot_id, name, region, spot_type, entrance_fee").execute().data
assert len(_rows) == 100, \
    f"100곳이 들어 있어야 합니다 (지금 {len(_rows)}곳) — 적재 셀을 다시 실행하세요"
assert len({r["region"] for r in _rows}) == 15, \
    "지역이 15종이어야 합니다 — region 열을 그대로 넣었는지 확인하세요"
assert len({r["spot_type"] for r in _rows}) == 5, \
    "유형이 5종이어야 합니다 — CSV 의 type 을 spot_type 으로 넣었는지 확인하세요"
assert all(isinstance(r["entrance_fee"], int) for r in _rows), \
    "입장료는 정수여야 합니다 — int() 로 바꿔 넣으세요"
assert sum(1 for r in _rows if r["entrance_fee"] == 0) == 82, \
    "무료인 곳이 82곳이어야 합니다 — 입장료를 제대로 옮겼는지 확인하세요"

# 벡터가 제대로 들어갔는지 실제 검색으로 되짚습니다.
# 임베딩을 빠뜨렸거나 '이름 + 설명' 이 아닌 다른 텍스트를 넣었다면 여기서 걸립니다.
_probe = supabase.rpc("match_spot", {
    "query_embedding": embed("바다를 보며 쉴 수 있는 곳"),
    "match_count": 3, "filter_region": None,
    "filter_type": None, "min_similarity": 0.0,
}).execute().data
assert len(_probe) == 3, f"검색이 3곳을 돌려줘야 합니다 (지금 {len(_probe)}곳)"
assert _probe[0]["name"] == "월정리 해변", \
    f"1위가 월정리 해변 여야 합니다 (지금 {_probe[0]['name']}) — " \
    "임베딩한 텍스트가 '이름 + 공백 + 설명' 이 맞는지 확인하세요"

# 이름으로도 찾히는지 — 설명만 임베딩했다면 여기서 다른 곳이 1위로 나옵니다.
_by_name = supabase.rpc("match_spot", {
    "query_embedding": embed("감천문화마을"),
    "match_count": 1, "filter_region": None,
    "filter_type": None, "min_similarity": 0.0,
}).execute().data
assert _by_name and _by_name[0]["name"] == "감천문화마을", \
    f"'감천문화마을' 로 찾았는데 1위가 " \
    f"{_by_name[0]['name'] if _by_name else '없음'} 입니다 — " \
    "이름을 빼고 설명만 임베딩한 것 같습니다('이름 + 공백 + 설명')"
print("✅ 통과!")

### 2단계. 뜻으로 찾고, 조건으로 좁히기

저장해 둔 `match_spot` 을 파이썬에서 부릅니다. `rpc` 의 첫 인자가 함수 이름, 둘째가 **인자 딕셔너리**이고, 실제 목록은 응답의 **`.data`** 에 들어 있습니다.

세 가지를 차례로 해 보세요.

1. `"바다를 보며 쉴 수 있는 곳"` — **전체에서** 3곳. **`hits_all`** 에 담으세요.
2. `"바닷가를 따라 산책하기 좋은 곳"` — **`부산` 지역 안에서만** 3곳. **`hits_region`** 에 담으세요.
3. `"전통시장에서 먹거리를 구경하고 싶어요"` — **`제주` 지역의 `미식`** 만 3곳. **`hits_both`** 에 담으세요.

담은 뒤 각각 이름·지역·유형·유사도를 한 줄씩 출력해 확인하세요.

**확인**: 1번의 1위는 **월정리 해변**(제주/해변), 2번은 3곳 모두 `부산` 이고 1위는 **광안리 해수욕장**, 3번은 모두 `제주`·`미식` 이고 1위는 **제주 동문시장** 입니다.

> 3번은 **2곳**만 나옵니다 — `제주` 의 `미식` 가 자료에 2곳뿐이기 때문입니다. `match_count` 는 "최대 몇 곳"이라는 뜻이지 "꼭 그만큼"이 아닙니다.

> 이렇게 **조건으로 좁히고(정형) 뜻으로 줄 세우는(벡터)** 것을 결합 검색이라고 합니다. 교안에서는 좁히는 열이 하나였지만, 여기서는 **둘을 겹쳐** 쓸 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 세 번 모두 같은 함수를 부른다. 달라지는 것은 질문과 두 필터 인자뿐이다
- 질문 문장을 그대로 넘기면 안 된다. 먼저 벡터로 바꿔야 한다
- 걸지 않을 필터에는 None 을 넘긴다
- 실행 결과 객체 자체가 아니라 그 안의 목록을 담아야 한다

세부구현:
1. 질문을 벡터로 바꾼다
2. 인자 다섯 개를 딕셔너리로 만들어 rpc 에 넘기고 실행한다
3. 응답에서 목록만 꺼내 변수에 담는다
4. 필터만 바꿔 두 번 더 되풀이한다
5. 반복문으로 이름과 지역, 유형, 유사도를 한 줄씩 출력한다
```

</details>

In [ ]:
# 질문을 벡터로 바꿔(embed) rpc 로 넘긴다 — 실제 목록은 응답의 .data 에 들어 있다
hits_all = supabase.rpc("match_spot", {
    "query_embedding": embed("바다를 보며 쉴 수 있는 곳"),
    "match_count": 3,
    "filter_region": None,
    "filter_type": None,
    "min_similarity": 0.0,
}).execute().data

# 지역만 좁힌다
hits_region = supabase.rpc("match_spot", {
    "query_embedding": embed("바닷가를 따라 산책하기 좋은 곳"),
    "match_count": 3,
    "filter_region": "부산",
    "filter_type": None,
    "min_similarity": 0.0,
}).execute().data

# 지역과 유형을 겹쳐 좁힌다
hits_both = supabase.rpc("match_spot", {
    "query_embedding": embed("전통시장에서 먹거리를 구경하고 싶어요"),
    "match_count": 3,
    "filter_region": "제주",
    "filter_type": "미식",
    "min_similarity": 0.0,
}).execute().data

for label, hits in [("전체", hits_all), ("지역", hits_region), ("지역+유형", hits_both)]:
    print(f"[{label}] {len(hits)}곳")
    for h in hits:
        print(f"  {h['similarity']:.3f}  {h['name']} ({h['region']}/{h['area']}/{h['spot_type']})")

<details><summary>해설</summary>

- 정렬은 데이터베이스가 이미 해 줍니다. 함수 안에서 `ORDER BY d.embedding <=> query_embedding` 으로 **거리순**으로 세워 오기 때문입니다.
- `.execute()` 는 응답 객체를 돌려줍니다. 실제 목록은 그 안의 **`.data`** 입니다 — 여기서 `.data` 를 빠뜨리면 뒤 단계가 전부 어긋납니다.
- 세 번째가 2곳뿐인 것은 **후보 자체가 그만큼**이기 때문입니다. 조건을 겹칠수록 후보가 빠르게 줄어듭니다 — 실무에서 필터를 너무 많이 걸면 "검색 결과 없음"이 자주 나오는 이유입니다.
- 좁히는 일은 SQL 함수 안의 `WHERE` 가 합니다. 우리는 값만 넘겼을 뿐인데 **조건 두 개와 벡터 정렬이 한 번의 왕복**으로 끝났습니다.

</details>

In [ ]:
# [자가채점]
for _name, _hits in [("hits_all", hits_all), ("hits_region", hits_region),
                     ("hits_both", hits_both)]:
    assert isinstance(_hits, list), f"{_name} 은 리스트여야 합니다 — .data 를 담으세요"
    assert _hits, f"{_name} 이 비어 있습니다"
    for key in ["name", "region", "area", "spot_type", "entrance_fee",
                "description", "similarity"]:
        assert key in _hits[0], f"{_name} 결과에 {key} 키가 없습니다: {sorted(_hits[0])}"
    _sims = [float(h["similarity"]) for h in _hits]
    assert _sims == sorted(_sims, reverse=True), \
        f"{_name} 이 유사도 높은 순이 아닙니다: {_sims}"

assert len(hits_all) == 3, f"hits_all 은 3곳이어야 합니다 (지금 {len(hits_all)}곳)"
assert hits_all[0]["name"] == "월정리 해변", \
    f"hits_all 1위는 월정리 해변 입니다: {hits_all[0]['name']}"

assert len(hits_region) == 3, f"hits_region 은 3곳이어야 합니다 (지금 {len(hits_region)}곳)"
assert {h["region"] for h in hits_region} == {"부산"}, \
    f"모두 부산 이어야 합니다 — filter_region 을 넘겼는지 확인하세요: " \
    f"{sorted({h['region'] for h in hits_region})}"
assert hits_region[0]["name"] == "광안리 해수욕장", \
    f"hits_region 1위는 광안리 해수욕장 입니다: {hits_region[0]['name']}"

assert len(hits_both) == 2, \
    f"hits_both 는 2곳입니다 — 후보가 그만큼뿐입니다 (지금 {len(hits_both)}곳)"
assert {(h["region"], h["spot_type"]) for h in hits_both} == {("제주", "미식")}, \
    "모두 제주 의 미식 여야 합니다 — 필터 두 개를 다 넘겼는지 확인하세요"
assert hits_both[0]["name"] == "제주 동문시장", \
    f"hits_both 1위는 제주 동문시장 입니다: {hits_both[0]['name']}"
print("✅ 통과!")

### 3단계. 검색을 함수로 묶기

2단계는 넘기는 값만 달랐습니다. **함수 하나**로 묶어 두면 문제 2 에서 그대로 씁니다.

**`search_spot(question, region=None, spot_type=None, k=3, min_similarity=0.0)`** 를 완성하세요. **`match_spot` 의 인자 다섯 개를 그대로 옮겨 온 모양**입니다.

- `question` — 사람이 쓴 질문(벡터로 바꾸는 일은 함수 안에서 합니다)
- `region` · `spot_type` — 좁힐 지역과 유형. **기본값은 둘 다 `None`**(안 좁힘)
- `k` — 최대 몇 곳. **기본값 3**
- `min_similarity` — 이 값보다 가까운 곳만. **기본값 `0.0`**(문턱 없음)
- **돌려주는 것**: `match_spot` 이 준 목록 그대로. **문턱을 넘긴 곳이 없으면 빈 리스트**가 됩니다.

만든 뒤 아래 세 가지로 직접 확인하세요.

1. `search_spot("조선 시대 왕이 살던 궁궐")` — 전체에서 3곳
2. `search_spot("전통시장에서 먹거리를 구경하고 싶어요", "제주", "미식")` — 좁혀서
3. `search_spot("노트북 배터리가 빨리 닳는데 어떻게 고치나요?", min_similarity=0.35)` — **빈 목록**

**확인**: 1번의 1위는 **경복궁**, 2번은 2곳 모두 `제주`·`미식`, 3번은 `[]` 입니다.

> **3번이 왜 빈 목록인가**: 여행과 아무 상관 없는 질문이라 가장 가까운 곳조차 유사도가 **0.15** 밖에 안 됩니다. 문턱 `0.35` 을 못 넘으니 아무것도 오지 않습니다. 벡터 검색은 **"가장 가까운 것"** 을 줄 뿐 **"충분히 가까운 것"** 을 주지 않기 때문에, 이 문턱이 없으면 엉뚱한 곳이 근거로 넘어갑니다. 문제 2 에서 쓰입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 2단계에서 쓴 호출을 그대로 함수 몸통으로 옮기고, 고정값이던 자리를 매개변수로 바꾼다
- 매개변수 다섯 개 모두 기본값이 있어야 부를 때 생략할 수 있다
- 문턱을 넘긴 곳이 없으면 데이터베이스가 알아서 빈 목록을 준다. 따로 처리할 것이 없다

세부구현:
1. 매개변수 다섯 개를 기본값과 함께 선언한다
2. 질문을 벡터로 바꾼다
3. 다섯 인자를 매개변수 값으로 채워 rpc 를 부른다
4. 목록을 return 한다
```

</details>

In [ ]:
# 2단계에서 고정값이던 자리를 매개변수로 바꾼다 — SQL 함수의 인자 다섯 개가 그대로 온다
def search_spot(question, region=None, spot_type=None, k=3, min_similarity=0.0):
    """질문과 뜻이 가까운 여행지를 최대 k 곳 찾아 돌려줍니다."""
    # 벡터로 바꾸는 일을 함수 안에 감춘다 - 쓰는 쪽은 문장만 넘기면 된다
    return supabase.rpc("match_spot", {
        "query_embedding": embed(question),
        "match_count": k,
        "filter_region": region,
        "filter_type": spot_type,
        "min_similarity": min_similarity,
    }).execute().data


for h in search_spot("조선 시대 왕이 살던 궁궐"):
    print(f"{h['similarity']:.3f}  {h['name']} ({h['region']}/{h['spot_type']})")

print("---")
for h in search_spot("전통시장에서 먹거리를 구경하고 싶어요", "제주", "미식"):
    print(f"{h['similarity']:.3f}  {h['name']}")

print("--- 여행과 무관한 질문:", search_spot("노트북 배터리가 빨리 닳는데 어떻게 고치나요?", min_similarity=0.35))

<details><summary>해설</summary>

- 기본값 덕분에 **전체 검색이 기본**이 되고, 좁히고 싶을 때만 값을 줍니다.
- 함수 안에서 `embed()` 를 부르므로 쓰는 쪽은 **문장만** 넘기면 됩니다. 이렇게 '벡터'라는 말이 밖으로 새어 나가지 않게 감추는 것이 좋은 함수의 조건입니다.
- 빈 목록을 만드는 일을 함수가 따로 하지 않습니다. 문턱을 넘긴 행이 없으면 데이터베이스가 0행을 돌려주고, 그것이 그대로 `[]` 가 됩니다.
- 이 함수 하나가 문제 2 의 재료가 됩니다 — RAG 의 **R**(찾기)이 완성됐습니다.

</details>

In [ ]:
# [자가채점]
import inspect

sig = inspect.signature(search_spot)
assert list(sig.parameters) == ["question", "region", "spot_type", "k",
                                "min_similarity"], \
    f"매개변수 이름과 순서를 확인하세요: {list(sig.parameters)}"
assert sig.parameters["region"].default is None, "region 의 기본값은 None 입니다"
assert sig.parameters["spot_type"].default is None, "spot_type 의 기본값은 None 입니다"
assert sig.parameters["k"].default == 3, "k 의 기본값은 3 입니다"
assert sig.parameters["min_similarity"].default == 0.0, \
    "min_similarity 의 기본값은 0.0 입니다"

r_all = search_spot("조선 시대 왕이 살던 궁궐")
assert len(r_all) == 3, f"기본 k 는 3 입니다 (지금 {len(r_all)}곳)"
assert r_all[0]["name"] == "경복궁", \
    f"1위는 경복궁 입니다: {r_all[0]['name']}"

r_both = search_spot("전통시장에서 먹거리를 구경하고 싶어요", "제주", "미식")
assert {(h["region"], h["spot_type"]) for h in r_both} == {("제주", "미식")}, \
    "두 필터를 모두 rpc 로 넘겼는지 확인하세요"

# 문턱값을 매개변수로 정말 전달하는지 봅니다.
assert search_spot("노트북 배터리가 빨리 닳는데 어떻게 고치나요?", min_similarity=0.35) == [], \
    "여행과 무관한 질문인데 결과가 왔습니다 — min_similarity 를 rpc 로 넘기세요"
assert search_spot("조선 시대 왕이 살던 궁궐", min_similarity=0.9) == [], \
    "min_similarity=0.9 인데 결과가 왔습니다 — 매개변수를 rpc 로 넘기지 않고 " \
    "고정값을 쓴 것 같습니다"
assert len(search_spot("조선 시대 왕이 살던 궁궐", min_similarity=0.0)) == 3, \
    "문턱 0.0 이면 걸러지지 않아야 합니다"
print("✅ 통과!")

---

# 문제 2. 근거로 추천하고 출처를 붙이기 (RAG 완성)

**배경**: 검색 결과를 그대로 보여 주면 사람이 세 곳을 읽고 스스로 고릅니다. 찾아온 곳들을 **근거로 삼아 LLM 이 한 문단으로 추천하게** 하면 훨씬 쓸모 있습니다.

다만 LLM 은 **모르는 것도 그럴듯하게 지어냅니다.** 그래서 세 가지를 지킵니다.

1. **근거 밖의 이야기는 하지 않게** 프롬프트로 못 박는다 — 자료에 없는 곳은 추천하지 않게.
2. **출처를 함께 보여 준다** — 어떤 곳을 근거로 삼았는지, 지역·유형·입장료는 얼마인지. 사람이 직접 확인할 수 있어야 합니다.
3. **근거가 아예 없으면 LLM 을 부르지 않는다** — 3단계에서 만든 문턱값이 여기서 쓰입니다. 빈 근거를 주고 "모른다고 답해" 라고 부탁하는 것보다, **애초에 묻지 않는 편**이 확실합니다.

> 아래 준비 셀부터는 **OpenAI 키**가 필요합니다. 문제 1 까지만 하고 마쳐도 됩니다.

In [ ]:
# [제공 코드] OpenAI 클라이언트 준비 — 이 셀은 실행만 하세요 (문제 2 부터 필요합니다).
# 14~16일차와 같은 방식입니다: .env 의 OPENAI_API_KEY 로 실제 OpenAI 에 연결합니다.
import os

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

# 키를 먼저 확인한다 — OpenAI() 를 만든 뒤에 검사하면 SDK 인증 오류가 먼저 나서 이 안내가 묻힌다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "문제 2 는 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 위 준비 셀부터 다시 실행하세요")

from openai import OpenAI

client = OpenAI()

# 추천할 만한 곳을 찾지 못했을 때 돌려줄 문구입니다 (문제 2 에서 글자 그대로 씁니다).
NO_ANSWER = "여행지 자료에서 관련된 곳을 찾지 못했습니다. 질문을 바꿔서 다시 물어봐 주세요."

print("OpenAI 클라이언트 준비 완료 — 실제 API 연결됨")

### 1단계. 근거를 한 덩어리 글로 묶기

LLM 에게 넘길 근거는 **글자 하나로 이어 붙인 문자열**이어야 합니다.

**`format_spots(hits)`** 를 완성하세요.

- `hits` — `search_spot()` 이 돌려준 목록
- **돌려주는 것**: 한 곳마다 **번호 · 이름 · 지역 · 유형 · 입장료 · 소개글**이 들어간 **하나의 문자열**. 곳과 곳 사이는 빈 줄로 띄웁니다.
- 번호는 **1 부터** 붙입니다 — LLM 이 "[1]" 처럼 근거를 가리킬 수 있게 하기 위해서입니다.

**예시 모양**

```text
[1] 경복궁 (서울/역사, 입장료 3000원)
    조선의 법궁으로 ...

[2] 덕수궁 (서울/역사, 입장료 1000원)
    ...
```

**확인**: `format_spots(search_spot("조선 시대 왕이 살던 궁궐"))` 안에 `[1]` · `[2]` · `[3]` 이 모두 있고, 각 여행지의 이름과 소개글이 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 조각을 리스트에 모아 두었다가 마지막에 하나로 잇는 방식이 읽기 쉽다
- 번호를 1 부터 세어 주는 반복 도구가 있다

세부구현:
1. 빈 리스트를 준비한다
2. hits 를 번호와 함께 돌며 한 곳의 글 조각을 만들어 담는다
3. 빈 줄을 사이에 두고 하나의 문자열로 잇는다
4. return 한다
```

</details>

In [ ]:
# 번호를 1부터 붙여 두면 LLM 에게 '근거 번호를 밝히라'고 요구할 수 있다
def format_spots(hits):
    """검색 결과를 LLM 에게 넘길 한 덩어리 글로 묶습니다."""
    parts = []
    for i, h in enumerate(hits, 1):
        parts.append(
            f"[{i}] {h['name']} ({h['region']}/{h['spot_type']}, "
            f"입장료 {h['entrance_fee']}원)\n    {h['description']}")
    return "\n\n".join(parts)


print(format_spots(search_spot("조선 시대 왕이 살던 궁궐")))

<details><summary>해설</summary>

- `enumerate(hits, 1)` 는 **1 부터** 번호를 세어 줍니다(기본은 0 부터).
- 문자열을 `+=` 로 계속 이어 붙이는 대신 리스트에 모아 `join` 하는 편이 빠르고 깔끔합니다.
- 지역·유형·입장료까지 함께 넣은 이유: **LLM 이 그 값들을 답에 쓸 수 있게** 하기 위해서입니다. 근거에 없는 값은 LLM 이 알 길이 없습니다.
- `hits` 가 빈 목록이면 결과는 빈 문자열입니다 — 2단계에서 그 경우를 따로 처리합니다.

</details>

In [ ]:
# [자가채점]
_hits = search_spot("조선 시대 왕이 살던 궁궐")
ctx = format_spots(_hits)
assert isinstance(ctx, str), "format_spots 는 문자열 하나를 돌려줘야 합니다"
for i in [1, 2, 3]:
    assert f"[{i}]" in ctx, f"[{i}] 번호가 없습니다 — 1 부터 번호를 붙이세요"
for h in _hits:
    assert h["name"] in ctx, f"근거에 이름이 빠졌습니다: {h['name']}"
    assert h["description"][:12] in ctx, f"근거에 소개글이 빠졌습니다: {h['name']}"
    assert h["region"] in ctx, f"근거에 지역이 빠졌습니다: {h['name']}"
    assert str(h["entrance_fee"]) in ctx, f"근거에 입장료가 빠졌습니다: {h['name']}"
print("✅ 통과!")

### 2단계. 근거로 추천하고, 없으면 물러서는 함수

이제 마지막 조각입니다. **`recommend(question, region=None, spot_type=None, k=3)`** 를 완성하세요.

**해야 할 일 (순서대로)**

1. `search_spot()` 으로 근거를 찾는다 — **`min_similarity` 에 `0.35`** 을 넘깁니다
2. **근거가 하나도 없으면, LLM 을 부르지 말고 준비 셀의 `NO_ANSWER` 를 그대로 `return`** 한다
3. `format_spots()` 로 근거를 한 덩어리 글로 묶는다
4. OpenAI 에 물어 답을 만든다 — 모델 `gpt-4o-mini`, `temperature=0`
   - **시스템 메시지**에 규칙을 적습니다: *주어진 근거의 장소만 추천할 것, 근거에 없는 곳은 지어내지 말 것, 두세 문장으로 짧게 답할 것*
   - **사용자 메시지**에 근거 글과 질문을 함께 담습니다
5. 답변 아래에 **출처**를 붙인다 — 근거마다 **이름 · 지역 · 권역(`area`) · 유형 · 입장료**
6. **답변과 출처가 이어 붙은 문자열 하나**를 `return` 한다

**확인**

- `recommend("조선 시대 왕이 살던 궁궐")` — 추천 문장이 있고, 그 아래에 근거 3곳의 **이름 · 지역 · 권역 · 유형 · 입장료**가 들어 있습니다.
- `recommend("노트북 배터리가 빨리 닳는데 어떻게 고치나요?")` — **`NO_ANSWER` 와 글자까지 똑같은 문자열**입니다. 출처도 붙지 않고, OpenAI 호출도 일어나지 않습니다.

> **2번이 오늘의 요점입니다.** 근거가 없을 때 LLM 에게 물어보면 "모른다"고 답할 때도 있고 그럴듯한 여행지를 지어낼 때도 있습니다 — **매번 다릅니다.** 아예 부르지 않으면 **언제나 같은 답**이 나가고, 돈과 시간도 쓰지 않습니다. 지킬 수 있는 규칙은 프롬프트가 아니라 **코드**에 두는 것이 낫습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 근거를 찾은 직후에 '비었으면 여기서 끝낸다'를 먼저 쓰고 시작하면 뒤가 단순해진다
- 나머지는 앞에서 만든 두 함수를 부르는 것으로 시작하면 몸통이 짧아진다
- LLM 호출은 14~16일차와 같은 모양이다. 역할과 내용을 담은 메시지 두 개를 넘긴다
- 출처 줄도 근거 목록을 돌며 만든다

세부구현:
1. 문턱값을 넘겨 검색해 근거를 찾는다
2. 근거가 비었으면 준비된 문구를 그대로 돌려주고 함수를 끝낸다
3. 남은 근거를 한 덩어리 글로 묶는다
4. 규칙을 적은 시스템 메시지와, 근거와 질문을 담은 사용자 메시지를 만든다
5. 모델을 부르고 답변 문자열을 꺼낸다
6. 근거를 돌며 이름과 지역, 권역, 유형, 입장료가 들어간 출처 줄들을 만든다
7. 답변과 출처를 이어 하나의 문자열로 돌려준다
```

</details>

In [ ]:
# RAG 의 마지막 조각 — 거르고 · 없으면 물러서고 · 묶고 · 묻고 · 출처를 붙인다
def recommend(question, region=None, spot_type=None, k=3):
    """찾아온 여행지를 근거로 추천하고 출처를 붙여 돌려줍니다."""
    # 1) 찾기 - 문턱값을 넘겨 엉뚱한 근거가 애초에 오지 않게 한다
    hits = search_spot(question, region, spot_type, k, min_similarity=0.35)

    # 2) 근거가 없으면 여기서 끝낸다 - 빈 근거로 LLM 을 부르면 지어낼 틈을 준다
    if not hits:
        return NO_ANSWER

    # 3) 묻기 - 시스템에는 규칙, 사용자에는 이번에 볼 근거와 질문을 담는다
    context = format_spots(hits)
    res = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system",
             "content": "너는 여행 안내원이다. 반드시 주어진 근거에 있는 장소만 추천하고, "
                        "근거에 없는 곳은 지어내지 마라. 두세 문장으로 짧게 답해라."},
            {"role": "user",
             "content": f"근거:\n{context}\n\n질문: {question}"},
        ],
    )
    reply = res.choices[0].message.content

    # 4) 출처 붙이기 - 사람이 직접 확인할 수 있어야 추천이 끝난다
    lines = ["", "출처"]
    for i, h in enumerate(hits, 1):
        lines.append(f"  [{i}] {h['name']} - {h['region']}({h['area']})/"
                     f"{h['spot_type']}, 입장료 {h['entrance_fee']}원")
    return reply + "\n" + "\n".join(lines)


print(recommend("조선 시대 왕이 살던 궁궐"))
print("=" * 60)
print(recommend("노트북 배터리가 빨리 닳는데 어떻게 고치나요?"))

<details><summary>해설</summary>

- **빈 근거를 걸러 내는 `if not hits` 한 줄**이 이 함수에서 가장 값진 줄입니다. 프롬프트로 부탁한 규칙은 지켜질 때도 있고 아닐 때도 있지만, 이 줄은 **언제나** 지켜집니다.
- 근거를 **시스템 메시지가 아니라 사용자 메시지**에 넣었습니다. 시스템에는 '어떻게 행동할지'(규칙), 사용자에는 '이번에 볼 자료와 질문'을 담는 것이 관례입니다.
- `temperature=0` 은 답을 최대한 일정하게 만듭니다. 그래도 **매번 완전히 같지는 않습니다** — 그래서 자가채점은 문장이 아니라 **구조**(출처가 붙었는가·거절했는가)를 봅니다.
- 출처에 입장료까지 넣은 이유: 추천을 받은 사람이 **바로 다음 판단**(갈지 말지)을 할 수 있어야 답변이 끝나기 때문입니다.

</details>

In [ ]:
# [자가채점]
import inspect

sig = inspect.signature(recommend)
assert list(sig.parameters) == ["question", "region", "spot_type", "k"], \
    f"매개변수는 question·region·spot_type·k 순서입니다: {list(sig.parameters)}"

# 1) 여행과 무관한 질문 — 문구가 글자까지 같아야 합니다(LLM 답이 섞이면 달라집니다).
out_none = recommend("노트북 배터리가 빨리 닳는데 어떻게 고치나요?")
assert out_none == NO_ANSWER, \
    "근거가 없는 질문인데 NO_ANSWER 를 그대로 돌려주지 않았습니다 — " \
    f"search_spot 에 min_similarity=0.35 을 넘기고, 비었으면 바로 return 하세요: " \
    f"{out_none[:60]}"

# 2) 근거가 있는 질문 — 출처가 붙고, 생성된 답변도 함께 있어야 합니다.
out = recommend("조선 시대 왕이 살던 궁궐")
assert isinstance(out, str), "recommend 는 문자열 하나를 돌려줘야 합니다"
src = search_spot("조선 시대 왕이 살던 궁궐", min_similarity=0.35)
assert len(src) == 3, "이 질문은 근거가 3곳 나와야 합니다"
for h in src:
    assert h["name"] in out, f"출처에 이름이 없습니다: {h['name']}"
    assert h["region"] in out, f"출처에 지역이 없습니다: {h['region']}"
    assert h["area"] in out, \
        f"출처에 권역이 없습니다: {h['area']} — match_spot 이 JOIN 으로 " \
        "붙여 준 area 를 쓰세요"
    assert h["spot_type"] in out, f"출처에 유형이 없습니다: {h['spot_type']}"
    assert str(h["entrance_fee"]) in out, f"출처에 입장료가 없습니다: {h['name']}"

# 출처만 있고 추천 문장이 없으면 안 됩니다 — 근거에서 그대로 가져온 글자를 모두 지워도
# 사람이 읽을 문장(LLM 이 만든 답)이 남아야 합니다.
body = out
for h in src:
    for piece in [h["name"], h["description"], h["region"], h["spot_type"]]:
        body = body.replace(piece, "")
body = "".join(ch for ch in body if not ch.isascii())   # 번호·괄호 같은 껍데기를 걷어냅니다
assert len(body.strip()) >= 30, \
    "생성된 추천 문장이 보이지 않습니다 — 출처만 붙이지 말고 LLM 의 답도 함께 돌려주세요"
print("✅ 통과!")

### 3단계. 세 가지 질문으로 시험해 보기

만든 추천 서비스를 **성격이 다른 질문 세 개**로 시험합니다.

1. `"바다를 보며 쉴 수 있는 곳"` — 전체에서 찾기
2. `"전통시장에서 먹거리를 구경하고 싶어요"` — `제주` 의 `미식` 로 좁혀서 찾기
3. `"노트북 배터리가 빨리 닳는데 어떻게 고치나요?"` — **여행과 아무 상관 없는 질문**

세 결과를 **`answers`** 라는 리스트에 순서대로 담고, 하나씩 출력해 확인하세요.

**보면서 생각할 것**

- 3번이 문턱값 없이 돌아갔다면 어떻게 됐을지 상상해 보세요. 가장 가까운 곳(유사도 0.15)이 근거랍시고 넘어가고, LLM 은 그것으로 무언가를 지어냈을 것입니다.
- 2번의 출처가 모두 같은 지역·유형인가요? 좁혀서 찾았기 때문입니다.
- 문턱값을 낮춰(예: 0.1) 3번을 다시 불러 보면 무엇이 달라지나요? **거절과 지어내기 사이의 손잡이가 바로 이 값**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문 세 개를 차례로 넣어 부르기만 하면 된다. 2번만 지역과 유형을 함께 넘긴다

세부구현:
1. 세 번 불러 결과를 리스트에 담는다
2. 구분선과 함께 하나씩 출력한다
```

</details>

In [ ]:
# 성격이 다른 세 질문으로 시험한다 - 마지막은 여행과 무관한 질문이다
answers = [
    recommend("바다를 보며 쉴 수 있는 곳"),
    recommend("전통시장에서 먹거리를 구경하고 싶어요", "제주", "미식"),
    recommend("노트북 배터리가 빨리 닳는데 어떻게 고치나요?"),
]

for a in answers:
    print(a)
    print("=" * 60)

<details><summary>해설</summary>

- 세 번째는 검색 단계에서 이미 걸러졌기 때문에 **LLM 이 아예 불리지 않습니다.** "모른다"는 답을 LLM 의 선의에 맡기지 않고 코드로 보장한 것입니다.
- 두 번째는 지역과 유형을 겹쳐 좁혔기 때문에 근거가 2곳뿐입니다. 그래도 추천은 성립합니다 — 근거가 적은 것과 없는 것은 다릅니다.
- 문턱값을 낮추면 답하는 질문이 늘지만 지어내기도 늘고, 높이면 그 반대입니다. 실무에서는 **답한 것 중 틀린 비율**과 **거절한 것 중 답할 수 있었던 비율**을 함께 재서 이 값을 정합니다.

</details>

In [ ]:
# [자가채점]
assert isinstance(answers, list) and len(answers) == 3, \
    f"answers 는 결과 3개의 리스트여야 합니다 (지금 {len(answers)}개)"
assert all(isinstance(a, str) and a.strip() for a in answers), \
    "세 결과 모두 비어 있지 않은 문자열이어야 합니다"

both_src = search_spot("전통시장에서 먹거리를 구경하고 싶어요", "제주", "미식", min_similarity=0.35)
assert both_src, "2번 질문은 근거가 나와야 합니다"
for h in both_src:
    assert h["name"] in answers[1], \
        f"2번 출처에 이 곳이 없습니다: {h['name']} — 지역과 유형을 넘겼는지 확인하세요"

assert answers[2] == NO_ANSWER, \
    "3번은 여행과 무관한 질문이라 NO_ANSWER 가 나와야 합니다 — 세 번째 질문을 확인하세요"
assert "월정리 해변" in answers[0], \
    "1번 출처에 월정리 해변 이 없습니다 — 첫 질문을 확인하세요"
assert len(set(answers)) == 3, "세 답이 모두 같습니다 — 질문을 각각 넘겼는지 확인하세요"
print("✅ 통과!")

## 다 풀었다면

- 자가채점이 모두 `✅ 통과!` 인지 확인하세요.
- 오늘 만든 것을 한 줄로 요약하면 이렇습니다: **소개글을 벡터로 바꿔 표에 담고 → 조건으로 좁히고 뜻으로 줄 세우고 → 문턱을 못 넘으면 물러서고 → 남은 근거로 LLM 이 추천하고 → 출처를 붙인다.**
- 교안에서는 연구비 FAQ 로 배웠고, 오늘은 **여행지**로 같은 일을 처음부터 해냈습니다. 데이터가 바뀌어도 절차는 그대로라는 것 — 그것이 오늘 확인한 것입니다.
- 표를 만들고(LV1) 이어 붙이고(LV2) 뜻으로 찾아 추천하기(LV3)까지 왔습니다. 다음 단원부터는 이 구조를 더 크게, 더 정확하게 키웁니다.